In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Bronze read karo
print("📖 Reading bronze crypto data...")
bronze_df = spark.table("financial_market.bronze.crypto")
print(f"Bronze records: {bronze_df.count():,}")

# Quality check
print("\n📊 Initial Data Quality Check:")
print(f"Null price_usd:   {bronze_df.filter(F.col('price_usd').isNull()).count()}")
print(f"Null crypto_id:   {bronze_df.filter(F.col('crypto_id').isNull()).count()}")
print(f"Null date:        {bronze_df.filter(F.col('date').isNull()).count()}")
print(f"Null market_cap:  {bronze_df.filter(F.col('market_cap').isNull()).count()}")
print(f"Null volume_24h:  {bronze_df.filter(F.col('volume_24h').isNull()).count()}")

display(bronze_df.limit(5))

In [0]:
print("✨ Starting cleaning pipeline...\n")

# Step 1 — null crypto_id remove karo
cleaned_df = bronze_df.filter(
    F.col("crypto_id").isNotNull() &
    (F.trim(F.col("crypto_id")) != "")
)
print(f"✅ After null crypto_id remove: {cleaned_df.count():,}")

# Step 2 — standardize crypto_id
cleaned_df = cleaned_df \
    .withColumn("crypto_id", F.lower(F.trim(F.col("crypto_id"))))

# Step 3 — invalid prices remove karo
initial = cleaned_df.count()
cleaned_df = cleaned_df.filter(
    F.col("price_usd").isNotNull() &
    (F.col("price_usd") > 0)
)
print(f"✅ After price validation: {cleaned_df.count():,}")

# Step 4 — null values handle karo
cleaned_df = cleaned_df.fillna(0, subset=["market_cap", "volume_24h"])

# Step 5 — derived columns add karo
cleaned_df = cleaned_df \
    .withColumn("market_cap_billions",
                F.round(F.col("market_cap") / 1e9, 2)) \
    .withColumn("volume_24h_millions",
                F.round(F.col("volume_24h") / 1e6, 2)) \
    .withColumn("is_stablecoin",
                F.col("crypto_id").isin(
                    "tether", "usd-coin", "dai", "true-usd"
                ))

# Step 6 — deduplicate by crypto_id + date
window_spec = Window \
    .partitionBy("crypto_id", "date") \
    .orderBy(F.col("fetch_timestamp").desc())

cleaned_df = cleaned_df \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

print(f"✅ After deduplication: {cleaned_df.count():,}")

# Step 7 — metadata add karo
cleaned_df = cleaned_df \
    .withColumn("processed_timestamp", F.current_timestamp()) \
    .withColumn("processing_date", F.current_date()) \
    .withColumn("data_quality", F.lit("clean"))

print(f"\n✅ Cleaning complete")
print(f"✅ Final row count: {cleaned_df.count():,}")
print(f"✅ Unique coins: {cleaned_df.select('crypto_id').distinct().count()}")
print(f"✅ Date range: {cleaned_df.agg(F.min('date')).collect()[0][0]} to {cleaned_df.agg(F.max('date')).collect()[0][0]}")

display(cleaned_df.limit(10))

In [0]:
print("💾 Saving to Silver layer...")

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("crypto_id") \
    .saveAsTable("financial_market.silver.crypto")

print(f"✅ Saved to: financial_market.silver.crypto")

# verify
df_verify = spark.sql("""
    SELECT
        crypto_id,
        COUNT(*)                     as total_rows,
        MIN(date)                    as earliest_date,
        MAX(date)                    as latest_date,
        ROUND(AVG(price_usd), 2)     as avg_price,
        ROUND(AVG(market_cap_billions), 2) as avg_mcap_billions
    FROM financial_market.silver.crypto
    GROUP BY crypto_id
    ORDER BY avg_mcap_billions DESC
""")

print(f"\n✅ Silver crypto verification:")
print(f"✅ Total rows: {spark.table('financial_market.silver.crypto').count():,}")
display(df_verify)

In [0]:
# # schema verify karo
# df = spark.table("financial_market.bronze.crypto")
# df.printSchema()
# # display(df.limit(5))